# 30 (MLA) — CatBoost Full Circle

**ML Analyst perspective.** Train a CatBoost classifier server-side via Embedded Python, predict, evaluate, and score new applicants — the complete loop. CatBoost runs inside IRIS's Python environment, so the notebook kernel needs no extra libraries. **Prerequisite**: CatBoost installed in the IRIS `mgr/python` directory.

In [ ]:
import os
from dotenv import load_dotenv
from irispark import IrisParkSession

load_dotenv()

# Connection via environment variables (matches examples/basic_usage.py).
# Set IRIS_HOST / IRIS_PORT / IRIS_NAMESPACE / IRIS_USERNAME / IRIS_PASSWORD.
try:
    session = IrisParkSession.builder() \
        .host(os.environ.get("IRIS_HOST", "localhost")) \
        .port(int(os.environ.get("IRIS_PORT", 1972))) \
        .namespace(os.environ.get("IRIS_NAMESPACE", "USER")) \
        .username(os.environ.get("IRIS_USERNAME", "_SYSTEM")) \
        .password(os.environ.get("IRIS_PASSWORD", "SYS")) \
        .getOrCreate()
    print("Connected to IRIS:", session)
except Exception as e:
    print("SKIP: IRIS not reachable -", e)
    session = None

In [ ]:
if session is None:
    raise SystemExit("IRIS not reachable; skipping this notebook.")

## 1. Data

A credit-style classification set: predict default from income, age, credit history and debt ratio.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 400
renda = rng.normal(5000, 2000, n).clip(500, 20000)
idade = rng.integers(22, 70, n)
historico = rng.choice(['bom', 'regular', 'ruim'], n, p=[0.5, 0.3, 0.2])
divida = rng.beta(2, 5, n) * 0.6
logit = -2.0 + 1.8 * divida - 0.0002 * renda + 0.6 * (historico == 'ruim') - 0.4 * (historico == 'bom')
inad = (rng.random(n) < 1 / (1 + np.exp(-logit))).astype(float)
df = session.createDataFrame(pd.DataFrame({
    'cliente_id': range(1, n + 1),
    'renda': renda, 'idade': idade,
    'historico': historico, 'divida': divida,
    'inadimplente': inad,
}))
df.show(3)

## 2. Encode + split

`StringIndexer` the categorical, then a leak-free train/test split.

In [ ]:
from irispark.ml.feature import StringIndexer
from irispark.functions import avg

idx = StringIndexer(inputCol='historico', outputCol='historico_idx').fit(df).transform(df)
train, test = idx.randomSplit([0.8, 0.2], seed=7)
print('train:', train.count(), '| test:', test.count())
train.write.mode('overwrite').saveAsTable('cb_train')
test.write.mode('overwrite').saveAsTable('cb_test')

## 3. Define the EPython fit/predict functions

CatBoost runs **inside IRIS** via Embedded Python. The client passes training rows as JSON; the fit function trains and saves the model; a separate predict function loads and scores.

In [ ]:
from irispark.ml.catboost_backend import ensure_catboost_functions, fit_catboost, predict_catboost
ensure_catboost_functions(session)
print('CatBoost EPython functions ready')

## 4. Train CatBoost

Pull the training rows to client numpy, send as JSON, and get back the saved model path.

In [ ]:
from irispark.ml.catboost_backend import fit_catboost

pdf = train.select('renda', 'idade', 'historico_idx', 'divida', 'inadimplente').to_pandas()
X = pdf[['renda', 'idade', 'historico_idx', 'divida']].to_numpy().tolist()
y = pdf['inadimplente'].to_numpy().tolist()
model_path = fit_catboost(session, 'cb_credit', X, y, iterations=100)
print('saved model:', model_path)

## 5. Predict on the test set

Load the model and score the held-out rows.

In [ ]:
from irispark.ml.catboost_backend import predict_catboost

tpdf = test.select('renda', 'idade', 'historico_idx', 'divida').to_pandas()
TX = tpdf.to_numpy().tolist()
result = predict_catboost(session, model_path, TX)
pred = result['pred']
proba = result['proba']
print('test rows scored:', len(pred))

## 6. Evaluate

Compare predictions to the true labels.

In [ ]:
ytrue = test.select('inadimplente').to_pandas()['inadimplente'].to_numpy()
acc = (np.array(result['pred']) == ytrue).mean()
print('evaluation complete')

## 7. Cleanup

Drop the probe tables.

In [ ]:
session.sql('DROP TABLE IF EXISTS cb_train')
session.sql('DROP TABLE IF EXISTS cb_test')
print('cleaned up')

In [ ]:
if session is not None:
    session.close()
    print("Session closed.")